In [1]:
!pip -q install google-generativeai

Ex 1 Prompt Chaining for Customer Support AI
Tools Used

Google Colab, Gemini API (google-generativeai)

Issue Classification

System Prompt:
A customer support triage assistant.
Return ONLY valid JSON with keys:

category

urgency (1–5)

facts (list)

suspected_cause

Goal: Classify issue and extract structured info.

Gather Missing Information

System Prompt:
Ask up to 3 clarifying questions.
Constraints:

Only questions

Friendly tone

1–2 lines each

Goal: Use structured output from Step 1.

Propose Solution + Escalation

System Prompt:
Provide:

Solution steps (bullet points)

Escalation rule: Escalate? Yes/No + reason

Be concise and professional.

Iteration Note

Initial classification prompt produced unstructured output.
Refined prompt to require strict JSON formatting.

In [1]:
!pip -q install google-genai

from google import genai
from google.colab import userdata

api_key = userdata.get("GEMINI_API_KEY")
if not api_key:
    raise ValueError("Missing GEMINI_API_KEY in Colab Secrets.")

client = genai.Client(api_key=api_key)

model = "gemini-flash-latest"

def call_llm(system_prompt, user_prompt):
    full_prompt = f"SYSTEM:\n{system_prompt}\n\nUSER:\n{user_prompt}"
    response = client.models.generate_content(
        model=model,
        contents=full_prompt
    )
    return response.text

Exercise 1 — Prompt Chaining for Customer Support AI
Tools Used

Google Colab

Gemini API (google-genai package)

Step 1 — Issue Classification (Structured Output)

Role: Customer support triage assistant
Task: Classify the issue and extract structured information.
Constraint: Output must be valid JSON only with keys:

category

urgency (1–5 scale)

facts (list)

suspected_cause

Purpose: Convert unstructured customer complaint into structured data for use in later steps.

Step 2 — Gather Missing Information

Role: Customer support agent
Task: Ask up to 3 clarifying questions.
Constraints:

Only questions

Friendly tone

1–2 lines each

Purpose: Use structured output from Step 1 to request any missing information needed to resolve the issue.

Step 3 — Propose Solution + Escalation Rule

Role: Customer support agent
Task: Provide:

Clear solution steps (bullet points)

Escalation decision: "Escalate? Yes/No" with reason

Purpose: Deliver resolution and determine whether escalation is required based on prior outputs.

Iteration Evidence

Initial classification prompt (Version 1) produced unstructured output.
The prompt was refined to require strict JSON formatting, improving clarity and enabling reliable chaining into Step 2.

In [2]:
client = genai.Client(api_key=api_key)

model = "gemini-flash-latest"

def call_llm(system_prompt, user_prompt):
    full_prompt = f"SYSTEM:\n{system_prompt}\n\nUSER:\n{user_prompt}"
    response = client.models.generate_content(
        model=model,
        contents=full_prompt
    )
    return response.text

In [3]:
import json

step1_system = (
    "Act as a customer support triage assistant.\n"
    "Return ONLY valid JSON with keys: category, urgency(1-5), facts(list), suspected_cause.\n"
    "No extra text outside JSON."
)

step1_user = "Customer says: 'My card was charged but the order failed and I got no confirmation email.'"

step1_output = call_llm(step1_system, step1_user)

print("STEP 1 OUTPUT (Structured JSON):\n")
print(step1_output)

data = json.loads(step1_output)

STEP 1 OUTPUT (Structured JSON):

{
  "category": "Billing & Payments",
  "urgency": 4,
  "facts": [
    "Customer's card was charged",
    "The order status was reported as failed during checkout",
    "No confirmation email was received"
  ],
  "suspected_cause": "Synchronization failure between the payment gateway and the order management system, resulting in a successful transaction capture without the creation of a corresponding order record."
}


In [4]:
# List available models for your API key/project
models = client.models.list()
for m in models:
    # print the model name/id
    print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-exp-image-generation
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-2.5-flash-lite-preview-09-2025
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-robotics-er-1.5-preview
models/gemini-2.5-computer-use-preview-10-2025
models/deep-research-pro-preview-12-2025
models/gemini-embedding-001
models/aqa
models/imagen-4.0-generate-001
models/imagen-

In [5]:
import json

step1_system = (
    "Act as a customer support triage assistant.\n"
    "Return ONLY valid JSON with keys: category, urgency(1-5), facts(list), suspected_cause.\n"
    "No extra text outside JSON."
)

step1_user = "Customer says: 'My card was charged but the order failed and I got no confirmation email.'"

step1_output = call_llm(step1_system, step1_user)

print("STEP 1 OUTPUT (Structured JSON):\n")
print(step1_output)

data = json.loads(step1_output)
print("\n✅ JSON parsed successfully.")

STEP 1 OUTPUT (Structured JSON):

{
  "category": "Billing & Payments",
  "urgency": 4,
  "facts": [
    "The customer's card was charged",
    "The order process failed",
    "No confirmation email was received"
  ],
  "suspected_cause": "Synchronization error between the payment gateway and the order management system."
}

✅ JSON parsed successfully.


In [7]:
step2_system = (
    "You are a customer support agent.\n"
    "Ask up to 3 clarifying questions.\n"
    "Constraints: ONLY questions, friendly tone, 1–2 lines each."
)

step2_user = (
    f"Category: {data['category']}\n"
    f"Facts: {data['facts']}\n"
    "Ask the minimum questions needed to resolve."
)

step2_output = call_llm(step2_system, step2_user)

print("STEP 2 OUTPUT (Questions):\n")
print(step2_output)

STEP 2 OUTPUT (Questions):

Could you please provide the email address you used during checkout so I can search for any record of the transaction?

Do you happen to remember the specific error message you saw when the order failed, or the exact date and time the charge occurred?

To help me track the payment, could you share the last four digits of the card that was charged?


In [8]:
step3_system = (
    "You are a customer support agent.\n"
    "Return in this exact format:\n"
    "Solution Steps:\n"
    "- ...\n"
    "- ...\n"
    "Escalation: Escalate? Yes/No — <one sentence reason>\n"
    "Keep it concise and professional."
)

step3_user = (
    f"Category: {data['category']}\n"
    f"Facts: {data['facts']}\n"
    "Customer answers:\n"
    "- Email used: student@example.com\n"
    "- Last 4 digits: 1234\n"
    "- Charge shows as 'Pending'\n"
    "- No confirmation email\n"
    "Provide final resolution and decide escalation."
)

step3_output = call_llm(step3_system, step3_user)

print("STEP 3 OUTPUT (Resolution):\n")
print(step3_output)

STEP 3 OUTPUT (Resolution):

Solution Steps:
- Verify the transaction status in the internal billing system using the email student@example.com and card ending in 1234.
- Inform the customer that the 'Pending' charge is a temporary authorization hold that will automatically expire since the order failed.
- Advise the customer to wait 3-5 business days for the funds to be released by their bank.
- Suggest clearing browser cache or using a different payment method to retry the order.

Escalation: Escalate? No — The 'Pending' status indicates an authorization hold that will resolve automatically once the bank processes the failed transaction.
